# Embedding readiness

Compares candidate embedding models on the real corpus chunks:

1. Speed and memory (this machine has ~7.4 GB RAM and a GPU via WSL).
2. Retrieval quality (recall@k, MRR) on a small question set, English and Filipino.

**Caveat:** the question set below is a starter. Relevance is decided by keywords found in the chunk, which is crude. Replace it with the golden set described in `docs/evaluation_rubric.md` before trusting small differences.

In [1]:
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "corpus").is_dir())  # repo root, wherever the notebook lives
CORPUS = REPO / "corpus"
sys.path.insert(0, str(REPO / "backend"))  # so `src.rag...` imports work

import hashlib


def unique_pdfs(folder):
    """PDFs in a folder, skipping byte-identical copies so nothing is embedded twice."""
    seen, pdfs = set(), []
    for path in sorted(folder.glob("*.pdf")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest not in seen:
            seen.add(digest)
            pdfs.append(path)
    return pdfs


PDFS = unique_pdfs(CORPUS)
[p.name for p in PDFS]

['20260914-NDRRMC-IRR-RA12287.pdf',
 'DISASTER PREPAREDNESS MANUAL.pdf',
 'ncrpasig-marikina-laguna-de-bay.pdf',
 'MMFMP.pdf']

In [2]:
import time
import numpy as np
import pandas as pd
import pymupdf
import torch
from sentence_transformers import SentenceTransformer

from src.rag.chunking import chunk_document
from src.rag.models import Document

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

chunks = []
for path in PDFS:
    with pymupdf.open(str(path)) as pdf:
        text = "\n\n".join(page.get_text() for page in pdf)
    document = Document(doc_id=path.stem, text=text, doc_type="manual", metadata={"source": path.name})
    chunks.extend(chunk_document(document))
texts = [c.content for c in chunks]
print(len(texts), "chunks")

/home/kenneth28/my_projects/Flood-Gentic/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


214 chunks


## Candidate models
E5 models need `query: ` / `passage: ` prefixes; bge-m3 does not. `multilingual-e5-large` is about the size of bge-m3, so the base model is included as a lighter option.

In [3]:
CANDIDATES = {
    # name: (hugging face id, query prefix, passage prefix)
    "bge-m3": ("BAAI/bge-m3", "", ""),
    "multilingual-e5-large": ("intfloat/multilingual-e5-large", "query: ", "passage: "),
    "multilingual-e5-base": ("intfloat/multilingual-e5-base", "query: ", "passage: "),
}
BATCH_SIZE = 16  # lower this if you run out of memory

## Question set (starter)
Each question lists keywords; a chunk counts as relevant if it contains any of them (case-insensitive). Edit freely.

In [4]:
QUESTIONS = [
    # (language, question, keywords that mark a relevant chunk)
    ("en", "What should I put in an emergency go-bag?", ["go bag", "go-bag", "emergency kit", "emergency supplies"]),
    ("en", "What should I do before a flood?", ["before a flood", "before flood", "flood preparedness"]),
    ("en", "Who should be contacted in an emergency?", ["hotline", "emergency contact", "911"]),
    ("en", "What are the functions of the NDRRMC?", ["ndrrmc"]),
    ("en", "How does flooding affect Metro Manila?", ["metro manila", "marikina"]),
    ("en", "How is disaster risk managed at the local level?", ["local government", "barangay", "lgu"]),
    ("tl", "Ano ang dapat ihanda bago dumating ang bagyo?", ["bagyo", "typhoon", "go bag", "emergency kit"]),
    ("tl", "Saan ako pupunta kapag may baha?", ["evacuation", "evacuate", "likas", "higher ground"]),
    ("tl", "Sino ang tatawagan kapag may sakuna?", ["hotline", "emergency contact", "911"]),
    ("tl", "Ano ang gagawin ng barangay sa panahon ng sakuna?", ["barangay"]),
]


def relevant_ids(keywords):
    """Indexes of chunks containing any keyword."""
    lowered = [k.lower() for k in keywords]
    return {i for i, t in enumerate(texts) if any(k in t.lower() for k in lowered)}


relevant = [relevant_ids(k) for _, _, k in QUESTIONS]
pd.DataFrame({"question": [q for _, q, _ in QUESTIONS], "relevant_chunks": [len(r) for r in relevant]})

,question,relevant_chunks
0,What should I put in an emergency go-bag?,7
1,What should I do before a flood?,0
2,Who should be contacted in an emergency?,10
3,What are the functions of the NDRRMC?,21
4,How does flooding affect Metro Manila?,16
5,How is disaster risk managed at the local level?,146
6,Ano ang dapat ihanda bago dumating ang bagyo?,28
7,Saan ako pupunta kapag may baha?,48
8,Sino ang tatawagan kapag may sakuna?,10
9,Ano ang gagawin ng barangay sa panahon ng sakuna?,60


Questions with 0 relevant chunks cannot be scored and are skipped in the metrics.

In [5]:
K_VALUES = (1, 5, 10)


def evaluate(scores_per_question, relevant_sets=None):
    """Return recall@k (share of queries with a hit in the top k) and MRR."""
    hits = {k: [] for k in K_VALUES}
    reciprocal_ranks = []
    for scores, rel in zip(scores_per_question, relevant_sets or relevant):
        if not rel:
            continue
        ranking = np.argsort(-scores)
        first = next(rank for rank, idx in enumerate(ranking, 1) if idx in rel)
        reciprocal_ranks.append(1 / first)
        for k in K_VALUES:
            hits[k].append(first <= k)
    result = {f"recall@{k}": round(float(np.mean(v)), 3) for k, v in hits.items()}
    result["mrr"] = round(float(np.mean(reciprocal_ranks)), 3)
    return result


results = []
for name, (model_id, query_prefix, passage_prefix) in CANDIDATES.items():
    print("running", name)
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    model = SentenceTransformer(model_id, device=DEVICE)

    start = time.perf_counter()
    passage_vectors = model.encode(
        [passage_prefix + t for t in texts], batch_size=BATCH_SIZE, normalize_embeddings=True
    )
    seconds = time.perf_counter() - start

    query_vectors = model.encode(
        [query_prefix + q for _, q, _ in QUESTIONS], normalize_embeddings=True
    )
    scores = query_vectors @ passage_vectors.T  # cosine similarity (vectors are normalized)

    row = {
        "model": name, "dim": passage_vectors.shape[1],
        "embed_seconds": round(seconds, 1),
        "chunks_per_second": round(len(texts) / seconds, 1),
        "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6) if DEVICE == "cuda" else None,
        **evaluate(scores),
    }
    # Per-language MRR, to see whether Filipino queries lag behind English.
    for lang in ("en", "tl"):
        idx = [i for i, (l, _, _) in enumerate(QUESTIONS) if l == lang]
        row[f"mrr_{lang}"] = evaluate([scores[i] for i in idx], [relevant[i] for i in idx])["mrr"]
    results.append(row)

    del model, passage_vectors, query_vectors
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

pd.DataFrame(results)

running bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7467.56it/s]

running multilingual-e5-large


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

running multilingual-e5-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  58%|█████▊    | 115/199 [00:00<00:00, 1140.11it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1603.50it/s]

,model,dim,embed_seconds,chunks_per_second,peak_gpu_mb,recall@1,recall@5,recall@10,mrr,mrr_en,mrr_tl
0,bge-m3,1024,26.3,8.1,3094,0.556,0.778,0.889,0.661,0.75,0.550
1,multilingual-e5-large,1024,17.3,12.4,2655,0.667,1.000,1.000,0.787,0.85,0.708
2,multilingual-e5-base,768,5.1,41.9,2248,0.667,0.889,0.889,0.752,0.84,0.643


## Reading the results
- `dim` must match `EMBEDDING_DIM` in `backend/src/rag/vector_store.py` (1024 for bge-m3 and e5-large, 768 for e5-base). Switching model means re-embedding everything.
- Prefer the model with the best recall@5 / MRR that still fits memory and embeds fast enough for re-ingestion.
- With only 10 keyword-labelled questions, differences under about 0.1 are noise.